# 06 — Historical CNN Model: EfficientNetB0 V3

> **Notebook status: reconstructed historical training record, not the original training notebook.** V3 was the model used before the V4 fine-tuning experiment. Its weights are not included in the final application repository. The V3 architecture, training settings, source hash and evaluation metrics are preserved as historical project evidence.

The current application uses V4. This notebook is retained to explain the model-development sequence and does not load a V3 checkpoint.

## Historical V3 model identity

- User-facing name at the time: **CNN**.
- Backbone: ImageNet-pretrained EfficientNetB0.
- Input: 256 × 256 × 3 RGB.
- Historical threshold: 0.47.
- Historical V3 checkpoint SHA-256: `ea47dc1e9ded13944a2ac866ae968f3956a003d9faaf39ce4a0dfa8a8cad65b5`.
- The V3 checkpoint was later used to initialise V4, but is not shipped in the final application repository.

## Recorded architecture

Input → augmentation → EfficientNetB0 (`include_top=False`, ImageNet weights) → GlobalAveragePooling2D → Dropout(0.40) → Dense(64, ReLU, L2=1e-4) → Dropout(0.30) → Dense(1, sigmoid).

Inference input is raw RGB float32 in the 0–255 range. EfficientNetB0 performs its own internal normalisation; do not divide by 255 externally.


In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd()
V3_RECORD = PROJECT_ROOT / "documentation" / "training_configuration.json"
V3_EVALUATION = PROJECT_ROOT / "documentation" / "v3_final_evaluation.json"
training_record = json.loads(V3_RECORD.read_text(encoding="utf-8"))
evaluation_record = json.loads(V3_EVALUATION.read_text(encoding="utf-8"))
print("Historical V3 backbone:", training_record["technical_backbone"])
print("Historical V3 SHA-256:", training_record["model_sha256"])
print("V3 test accuracy:", evaluation_record["test"]["accuracy"])
print("V3 checkpoint bundled:", (PROJECT_ROOT / "models_v3" / "car_cnn_model_v3.keras").is_file())
print("The V3 weights are intentionally excluded; V4 is the only model shipped for inference.")


## Training data coverage

| Split | Expected candidates | Loaded by the V3 Keras directory loader | Skipped WebP files |
|---|---:|---:|---:|
| Training | 874 | 872 | 2 |
| Validation | 188 | 185 | 3 |
| Testing | 188 | 186 | 2 |

The V3 metrics use these smaller effective counts. The full split remains 1,250 candidates, but the model run did not load all of them. This limitation must remain visible in any explanation or report.


## Two-phase training procedure

### Phase 1 — Frozen backbone

- Use ImageNet-pretrained EfficientNetB0.
- Freeze the backbone and train the classification head only.
- Optimizer: Adam.
- Learning rate: 0.001.
- Maximum epochs: 15.

### Phase 2 — Fine-tuning

- Unfreeze the top 40 backbone layers.
- Keep all Batch Normalization layers frozen.
- Optimizer: Adam.
- Learning rate: 0.00001.
- Maximum epochs: 25.

Other recorded techniques: binary cross-entropy with label smoothing 0.05, data augmentation, dropout, L2 regularisation, early stopping and learning-rate reduction. Exact callback patience/factor settings and batch size were not found in the recovered files, so they are not guessed here.

The model configuration records training seed 42 and dataset split seed 211. The test split was not used during training or threshold tuning.


In [ ]:
# Optional architecture reconstruction for study only. This cell is not run automatically.
# It may download ImageNet weights if RUN_ARCHITECTURE_BUILD is changed to True.
RUN_ARCHITECTURE_BUILD = False
if RUN_ARCHITECTURE_BUILD:
    import tensorflow as tf
    from tensorflow.keras import layers, regularizers, Model
    inputs = tf.keras.Input(shape=(256, 256, 3))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.04)(x)
    x = layers.RandomZoom(0.10)(x)
    x = layers.RandomContrast(0.10)(x)
    x = layers.RandomBrightness(0.10, value_range=(0, 255))(x)
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(256, 256, 3)
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    reconstructed_model = Model(inputs, outputs, name="car_authenticity_effnet")
    reconstructed_model.summary()
else:
    print("Architecture build is disabled. This notebook does not download weights or train a model by default.")


## Training workflow (settings known; callback details incomplete)

The following is a run-plan, not a ready-to-execute retraining command. It is shown so a reviewer can follow the two training phases. To reproduce the original run exactly, the missing original training script, callback values, batch size, dataset loader arguments and training logs are still required. No retraining is performed as part of this package revision.

```python
# Phase 1
backbone.trainable = False
compile(model, optimizer=Adam(learning_rate=0.001),
        loss=BinaryCrossentropy(label_smoothing=0.05))
fit(model, training_data, validation_data=validation_data, epochs=15,
    callbacks=[early_stopping, reduce_learning_rate])

# Phase 2
backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False
compile(model, optimizer=Adam(learning_rate=0.00001),
        loss=BinaryCrossentropy(label_smoothing=0.05))
fit(model, training_data, validation_data=validation_data, epochs=25,
    callbacks=[early_stopping, reduce_learning_rate])
```

The `callbacks`, exact batch size and data-loading arguments above are intentionally not instantiated because their historical values are not present in the recovered source. This avoids silently creating a different experiment and claiming it is the original run.